Tools allow Claude to access information from the outside world, extending its capabilities beyond what it learned during training. By default, Claude only knows information from its training data and can't access current events, real-time data, or external systems. Tool use solves this limitation by creating a structured way for Claude to request and receive fresh information.

![](../images/img15.png)

In [1]:
from google import genai
from dotenv import load_dotenv
import os


load_dotenv(override=True)
API_KEY = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=API_KEY)

In [2]:
# Helper functions
def add_user_message(messages, text):
    messages.append({
        "role": "user",
        "parts": [{"text": text}]
    })

def add_model_message(messages, text):
    messages.append({
        "role": "model",
        "parts": [{"text": text}]
    })

def chat(messages, response_schema=None, system_prompt=None, temperature=1.0):
    response = client.models.generate_content(
        model="gemini-flash-latest",
        contents=messages,
        config=genai.types.GenerateContentConfig(
            system_instruction=system_prompt,
            max_output_tokens=5000,
            temperature=temperature,
            response_mime_type="application/json" if response_schema else None,
            response_schema=response_schema
        )
    )
    return response.text

In [3]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

In [ ]:
from google.genai import types

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


# Gemini uses types.Tool and types.FunctionDeclaration instead of ToolParam
get_current_datetime_schema = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="get_current_datetime",
            description="Returns the current date and time formatted according to the specified format string. Uses Python's strftime formatting directives.",
            parameters=types.Schema(
                type="OBJECT",
                properties={
                    "date_format": types.Schema(
                        type="STRING",
                        description="A strftime-compatible format string that controls the output format. For example, '%Y-%m-%d' returns '2026-05-18', '%H:%M:%S' returns '14:30:00', and '%Y-%m-%d %H:%M:%S' returns '2026-05-18 14:30:00'. Must not be empty.",
                    )
                },
                required=[]
            )
        )
    ]
)

In [21]:
messages = [
    {
        "role": "user",
        "parts": [
            {"text": "What is the exact time formatted as HH:MM:SS"}
        ]
    }
]

response = client.models.generate_content(
    model="gemini-flash-latest",
    contents=messages,
    config={
        "tools": [get_current_datetime_schema]
    }
)

print(response)


messages.append({
    "role": "model",
    "parts": [response.candidates[0].content.parts[0]]
})



sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        function_call=FunctionCall(
          args={
            'date_format': '%H:%M:%S'
          },
          id='0zxiaiwg',
          name='get_current_datetime'
        ),
        thought_signature=b'\x12\xf2\x04\n\xef\x04\x01\x0c9\xd6\xc7C\xcc\x10\x7f\x99#\xfb\x88\xf1\x11\x94C\x88$\xa3B\xbd-\x93\xec\x0f{\x9d\xa1g\xe7\xc2\x90y\x0f\x13\x10\xc7a\x0f\x87\x0e- \xf7c1\xc5fC\x19\xb9\x9cV\xa6\xca\x7f\xc9\xed\x18\xc6\xa75c\xc7\x14X\xc6\x9ch\xc6\x8e<S\x01\x0c(\x9eyFvh\x9a\xa8U\xad=\xde\xc9D|...'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3-flash-preview' prompt_feedback=None response_id='axELaoLoGI-0juMP5MvY6AQ' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=25,
  prompt_token_count=171,
  prompt_tokens_details=[
    ModalityTokenCount(
      mod

In [24]:
result = get_current_datetime(**response.candidates[0].content.parts[0].function_call.args)

In [26]:
fc = response.candidates[0].content.parts[0].function_call

messages.append({
    "role": "user",
    "parts": [
        types.Part(
            function_response=types.FunctionResponse(
                name=fc.name,
                response={"result": result}
            )
        )
    ]
})

In [28]:
response = client.models.generate_content(
    model="gemini-flash-latest",
    contents=messages,
    config=types.GenerateContentConfig(
        tools=[get_current_datetime_schema] 
    )
)

print(response.text)

The exact time is 19:10:43.
